In [ ]:
!pip install -q kagglehub

In [ ]:
import os

for root, dirs, files in os.walk(path):
    print(root)
    break

In [ ]:
train_dir = "/kaggle/input/chest-xray-pneumonia/chest_xray/train"
val_dir   = "/kaggle/input/chest-xray-pneumonia/chest_xray/val"
test_dir  = "/kaggle/input/chest-xray-pneumonia/chest_xray/test"

In [ ]:
import os

for root, dirs, files in os.walk(path):
    print(root)
    if len(files) > 0:
        print("Files:", files[:5])
    print("-"*50)

In [ ]:
def count_images(folder):
    total = 0
    for cls in os.listdir(folder):
        cls_path = os.path.join(folder, cls)
        if os.path.isdir(cls_path):
            total += len(os.listdir(cls_path))
    return total

print("Train:", count_images(train_dir))
print("Validation:", count_images(val_dir))
print("Test:", count_images(test_dir))

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import random

classes = ['NORMAL','PNEUMONIA']

plt.figure(figsize=(12,6))

for i in range(6):

    cls=random.choice(classes)

    img_path=os.path.join(
        train_dir,
        cls,
        random.choice(os.listdir(os.path.join(train_dir,cls)))
    )

    img=Image.open(img_path)

    plt.subplot(2,3,i+1)
    plt.imshow(img,cmap='gray')
    plt.title(cls)
    plt.axis('off')

plt.show()

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = 224
BATCH_SIZE = 32

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

In [ ]:
test_datagen = ImageDataGenerator(
    rescale=1./255
)

In [ ]:
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224,224),
    batch_size=32,
    class_mode='binary'
)

val_generator = test_datagen.flow_from_directory(
    val_dir,
    target_size=(224,224),
    batch_size=32,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(224,224),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *

cnn_model = Sequential()

cnn_model.add(
    Conv2D(
        32,
        (3,3),
        activation='relu',
        input_shape=(224,224,3)
    )
)

cnn_model.add(MaxPooling2D())

cnn_model.add(
    Conv2D(
        64,
        (3,3),
        activation='relu'
    )
)

cnn_model.add(MaxPooling2D())

cnn_model.add(
    Conv2D(
        128,
        (3,3),
        activation='relu'
    )
)

cnn_model.add(MaxPooling2D())

cnn_model.add(Flatten())

cnn_model.add(Dense(128,activation='relu'))

cnn_model.add(Dropout(0.5))

cnn_model.add(Dense(1,activation='sigmoid'))

cnn_model.summary()

In [ ]:
cnn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
import time

start=time.time()

history_cnn = cnn_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10
)

cnn_time=time.time()-start

print("CNN Training Time:",cnn_time)

In [ ]:
from sklearn.metrics import *

pred_prob = cnn_model.predict(test_generator)

pred = (pred_prob > 0.5).astype(int)

y_true = test_generator.classes

print(classification_report(y_true,pred))

In [ ]:
import seaborn as sns

cm = confusion_matrix(y_true,pred)

plt.figure(figsize=(6,6))

sns.heatmap(
    cm,
    annot=True,
    fmt='d'
)

plt.title("CNN Confusion Matrix")

plt.show()